# Digital Contract Hub 
Notebook này chứa TẤT CẢ mã nguồn chạy trực tiếp trên các cell. Nó sẽ đọc các file PDF thật trong thư mục `data`, bóc tách, đưa vào VectorDB và hỏi đáp.

In [89]:
import os
import json
from typing import List
from dotenv import load_dotenv
from datasets import Dataset

from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import AIMessage, HumanMessage 
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever
from langchain_cohere import CohereRerank

In [90]:
load_dotenv(override=True)

def test_gemini_api():
    api_key = os.getenv("GEMINI_API_KEY")
    
    try:
        llm = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash", 
            google_api_key=api_key
        )

        print("--- Đang kết nối tới Google Gemini 2.5 Flash ---")
        time.sleep(1) 
        
        response = llm.invoke([HumanMessage(content="Xin chào!")])
        print(f"Kết nối thành công! Gemini trả lời: {response.content}")
        
    except Exception as e:
        print(f"{e}")

if __name__ == "__main__":
    test_gemini_api()

--- Đang kết nối tới Google Gemini 2.5 Flash ---
Kết nối thành công! Gemini trả lời: Chào bạn! Tôi có thể giúp gì cho bạn?


# 1. Xác định các 'element' trong file PDF

In [91]:
def partition_documents(file_path: str):
  print(f'Partition documents from {file_path}...')
  
  elements = partition_pdf(
    filename=file_path,
    strategy='hi_res', # Độ phân giải cao, cần cài dependency Tesseract OCR
    infer_table_structure=True, # Mục đích: Tạo attr 'text_as_html' => LLM hiểu cấu trúc bảng
    extract_image_block_types=['Image'], # Tách ảnh
    extract_image_block_to_payload=True # Tạo chuỗi Base64 => LLM hiểu cấu trúc ảnh
  )
  print(f'Extract {len(elements)} elements')

  return elements
file_path = 'data/example.pdf'
elements = partition_documents(file_path)

No languages specified, defaulting to English.


Partition documents from data/example.pdf...
Extract 27 elements


In [92]:
# Các dạng Elements
set([type(el) for el in elements])

{unstructured.documents.elements.ListItem,
 unstructured.documents.elements.NarrativeText,
 unstructured.documents.elements.Table,
 unstructured.documents.elements.Text,
 unstructured.documents.elements.Title}

In [93]:
for el in elements:
  print(f'Element type: {type(el)}, text: {el.text[:100]}...')

Element type: <class 'unstructured.documents.elements.Title'>, text: LOGISTICS SERVICES AGREEMENT...
Element type: <class 'unstructured.documents.elements.Title'>, text: Reference Number: ONPOINT-FFL-2025-089...
Element type: <class 'unstructured.documents.elements.NarrativeText'>, text: This Logistics Services Agreement (the "Agreement") is entered into and made effective as of June 01...
Element type: <class 'unstructured.documents.elements.Title'>, text: PARTY A: ONPOINT SERVICE JOINT STOCK COMPANY...
Element type: <class 'unstructured.documents.elements.NarrativeText'>, text: Address: 4th Floor, Sailing Tower, 111A Pasteur, Ben Nghe Ward, District 1, Ho Chi Minh City, Vietna...
Element type: <class 'unstructured.documents.elements.Title'>, text: PARTY B: FASTFORWARD LOGISTICS Cco., LTD....
Element type: <class 'unstructured.documents.elements.NarrativeText'>, text: Address: Plot 24, VSIP I Industrial Park, Thuan An City, Binh Duong Province, Vietnam....
Element type: <class 'unstru

# 2. Tạo các 'chunk' để chuẩn bị lưu vào Database

In [94]:
def create_chunk_by_title(elements):
  print(f"Chunking Documents...")
  chunks = chunk_by_title(
    elements=elements,
    combine_text_under_n_chars=250,      
    max_characters=1500,                
    new_after_n_chars=1000,                
    overlap=150                          
)
  print(f"Succesfully created {len(chunks)} chunks")
  return chunks

chunks = create_chunk_by_title(elements)

Chunking Documents...
Succesfully created 8 chunks


# 3. Hàm xác định những 'chunk' nào có chứa các bảng số liệu

In [95]:
def seperate_content_types(chunk): # Xác định xem chunk gồm những element nào
  content_data = {
    'text': chunk.text,
    'tables': [],
    'types': ['text']
  }
  # Kiểm tra xem trong chunk có các thuộc tính cần xét không
  if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
    for element in chunk.metadata.orig_elements:
      element_type = type(element).__name__
      if element_type == 'Table':
        if 'table' not in content_data['types']:
          content_data['types'].append('table')
        table_html = getattr(element.metadata, 'text_as_html', element.text) 
        content_data['tables'].append(table_html)
  return content_data

# 4. Hàm viết lại những 'chunk' trên bằng LLM

In [96]:
def create_ai_enhanced_summary(text: str, tables: List[str]):
  print('Enhance content before embedding...')

  llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash', 
    google_api_key=os.getenv('GEMINI_API_KEY'),
    temperature=0.1
)
  table_content = "\n\n".join([f'Table (html):{t}' for t in tables])
  prompt_text = f"""
  You are an analyst. Summarize this document for research purposes.
  REQUIREMENT: Extract data from the tabl.
  TEXT: {text}
  {table_content}
  """
  content = [
    {
      'type': 'text',
      'text': prompt_text
    }
  ]
  message = HumanMessage(content=content)
  response = llm.invoke([message])
  return response.content

# 5. Tổng hợp 2 hàm trên

In [97]:
def summarise_chunks(chunks):
  langchain_documents = []
  total_chunks = len(chunks)
  
  # STATEFUL TRACKING: Giữ tiêu đề ở ngoài vòng lặp
  current_title = "General Clause"
  
  for i, chunk in enumerate(chunks):
    print(f'-> Processing chunk {i + 1}/{total_chunks}')
    content_data = seperate_content_types(chunk)
    
    # Lấy Tên Điều Khoản
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
      for el in chunk.metadata.orig_elements:
        if type(el).__name__ == 'Title':
          current_title = el.text # Cập nhật title mới
          break
          
    clause_name = current_title
    
    enhanced_content = content_data['text']
    if content_data['tables']:
      try:
        # TRUYỀN THÊM THAM SỐ IMAGES ĐỂ TRÁNH LỖI (dù bạn đã xóa logic xử lý ảnh bên trong)
        enhanced_content = create_ai_enhanced_summary(
          content_data['text'],
          content_data['tables'],
        )
      except TypeError:
        # Nếu hàm tạo ra lỗi TypeError do chỉ nhận 2 tham số, ta gọi lại với 2 tham số
        try:
            enhanced_content = create_ai_enhanced_summary(
              content_data['text'],
              content_data['tables']
            )
        except Exception as e2:
            print(f"Failed {e2} !")
            enhanced_content = content_data['text']
      except Exception as e:
        print(f"Failed {e} !")
        enhanced_content = content_data['text']

    doc = Document(
      page_content=enhanced_content,
      metadata={
          'source': 'data/example.pdf',
          'clause': clause_name,
          'page': chunk.metadata.page_number if hasattr(chunk.metadata, 'page_number') else 1
      }
    )
    langchain_documents.append(doc)
  print(f"Finished ! Process {len(langchain_documents)} chunks")
  return langchain_documents

processed_chunks = summarise_chunks(chunks)

-> Processing chunk 1/8
-> Processing chunk 2/8
-> Processing chunk 3/8
-> Processing chunk 4/8
-> Processing chunk 5/8
-> Processing chunk 6/8
Enhance content before embedding...
-> Processing chunk 7/8
-> Processing chunk 8/8
Finished ! Process 8 chunks


In [98]:
processed_chunks[0]

Document(metadata={'source': 'data/example.pdf', 'clause': 'LOGISTICS SERVICES AGREEMENT', 'page': 1}, page_content='LOGISTICS SERVICES AGREEMENT\n\nReference Number: ONPOINT-FFL-2025-089\n\nThis Logistics Services Agreement (the "Agreement") is entered into and made effective as of June 01, 2025 (the "Effective Date"), by and between the following corporate entities:\n\nPARTY A: ONPOINT SERVICE JOINT STOCK COMPANY\n\nAddress: 4th Floor, Sailing Tower, 111A Pasteur, Ben Nghe Ward, District 1, Ho Chi Minh City, Vietnam. Represented by: Logistics Operations Director')

# 6. Lưu vào Vector Database (ChromaDB)

In [99]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import os

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2", 
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True} 
)

db_path = "./chroma_db"

print("Lưu vào Vector Database (ChromaDB)...")
vector_db = Chroma.from_documents(documents=processed_chunks, embedding=embedding_model, persist_directory=db_path)
print("Hoàn tất lưu trữ VectorDB!")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13010.10it/s]


Lưu vào Vector Database (ChromaDB)...
Hoàn tất lưu trữ VectorDB!


# 7. Vector Search & LLM Trả lời (RAG)

In [100]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
import os

# Khởi tạo LLM chung
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.1
)

retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# Câu hỏi của người dùng
user_query = "Nếu số đơn hàng là 60000 thì chi phí là bao nhiêu"
print(f"Câu hỏi gốc (Tiếng Việt): '{user_query}'")

# QUERRY TRANSLATION (Dịch câu hỏi sang Tiếng Anh)
translation_prompt = f"Translate the following question into English. Return ONLY the English translation, nothing else.\nQuestion: {user_query}"
search_query = llm.invoke([HumanMessage(content=translation_prompt)]).content.strip()
print(f"Đã dịch sang Tiếng Anh để tìm kiếm: '{search_query}'\n")

docs = retriever.invoke(search_query)

context_text = ""
print(f"--- ĐOẠN TRÍCH ĐƯỢC TÌM THẤY TỪ DB ---")
for idx, doc in enumerate(docs):
    source = doc.metadata.get('source', 'Unknown')
    page = doc.metadata.get('page', 'Unknown')
    clause = doc.metadata.get('clause', 'Unknown')
    
    print(f"[{idx+1}] File: {source} | Trang: {page} | Điều khoản: {clause}")
    context_text += f"\n--- Trích đoạn {idx+1} [Nguồn: {source} | Trang: {page} | Điều khoản: {clause}] ---\n{doc.page_content}\n"

# LLM Trả lời bằng Tiếng Việt (Generation)
prompt_text = f"""Bạn là một trợ lý pháp lý AI. 
Dựa TRỰC TIẾP vào các đoạn trích hợp đồng dưới đây, hãy trả lời câu hỏi của người dùng.
NẾU TRONG ĐOẠN TRÍCH KHÔNG CÓ THÔNG TIN, HÃY TRẢ LỜI LÀ "Không có thông tin", TUYỆT ĐỐI KHÔNG ĐƯỢC BỊA ĐẶT.
Khi trả lời, BẮT BUỘC phải ghi rõ nguồn trích dẫn theo định dạng: "Nguồn: [File] | Trang: [Trang] | Điều khoản: [Điều khoản]".
QUAN TRỌNG: Hãy trả lời bằng TIẾNG VIỆT.

CÂU HỎI: {user_query}

ĐOẠN TRÍCH:
{context_text}
"""

response = llm.invoke([HumanMessage(content=prompt_text)])
print(f"\nTRỢ LÝ TRẢ LỜI:\n{response.content}")


Câu hỏi gốc (Tiếng Việt): 'Nếu số đơn hàng là 60000 thì chi phí là bao nhiêu'
Đã dịch sang Tiếng Anh để tìm kiếm: 'What is the cost if the number of orders is 60,000?'

--- ĐOẠN TRÍCH ĐƯỢC TÌM THẤY TỪ DB ---
[1] File: data/example.pdf | Trang: 2 | Điều khoản: 3. SERVICE FEES & PRICING MATRIX
[2] File: data/example.pdf | Trang: 2 | Điều khoản: 3. SERVICE FEES & PRICING MATRIX
[3] File: data/example.pdf | Trang: 1 | Điều khoản: 2, SERVICE LEVEL AGREEMENTS (SLA)

TRỢ LÝ TRẢ LỜI:
Nếu số đơn hàng là 60000, chi phí sẽ được áp dụng theo mức "Over 50,000 orders" (Trên 50.000 đơn hàng) như sau:

*   **Phí vận chuyển tiêu chuẩn / đơn vị:** 19,000 VND
*   **Phí lưu kho hàng tháng / pallet tiêu chuẩn:** Miễn phí

Nguồn: data/example.pdf | Trang: 2 | Điều khoản: 3. SERVICE FEES & PRICING MATRIX
